# Day 28 — Point-in-Time Economic Backtesting

## Objectives
- Retrieve historical economic data with availability dates
- Align economic releases with eligible trading sessions
- Prevent look-ahead bias
- Compare portfolio performance across economic conditions

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

conn = sqlite3.connect("hedge_fund.db")

tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", conn)

display(tables)

for table_name in [
    "daily_prices",
    "economic_vintages"
]:
    print(f"\n{table_name}")

    columns = pd.read_sql_query(
        f"PRAGMA table_info({table_name})",
        conn
    )

    display(columns[["name", "type"]])

    sample = pd.read_sql_query(
        f"SELECT * FROM {table_name} LIMIT 5",
        conn
    )

    display(sample)

,name
0,daily_prices
1,derived_economic_indicators
2,economic_data
3,economic_vintage_snapshots
4,economic_vintages
5,portfolio_holdings
6,portfolio_weights



daily_prices


,name,type
0,date,TEXT
1,ticker,TEXT
2,close_price,REAL


,date,ticker,close_price
0,2025-09-23,AAPL,253.493591
1,2025-09-24,AAPL,251.381409
2,2025-09-25,AAPL,255.924637
3,2025-09-26,AAPL,254.519806
4,2025-09-29,AAPL,253.493591



economic_vintages


,name,type
0,indicator,TEXT
1,observation_date,TEXT
2,available_date,TEXT
3,vintage_end_date,TEXT
4,value,REAL
5,source,TEXT


,indicator,observation_date,available_date,vintage_end_date,value,source
0,CPI,2024-01-01,2024-02-13,2025-02-11,309.685,ALFRED
1,CPI,2024-01-01,2025-02-12,2026-02-12,309.794,ALFRED
2,CPI,2024-01-01,2026-02-13,2026-09-24,309.698,ALFRED
3,CPI,2024-02-01,2024-03-12,2025-02-11,311.054,ALFRED
4,CPI,2024-02-01,2025-02-12,2026-02-12,311.022,ALFRED


In [2]:
coverage = pd.read_sql_query("""
SELECT
    indicator,
    COUNT(*) AS records,
    MIN(observation_date) AS first_observation,
    MAX(observation_date) AS latest_observation,
    MIN(available_date) AS first_available_date,
    MAX(available_date) AS latest_available_date
FROM economic_vintages
GROUP BY indicator
ORDER BY indicator
""", conn)

display(coverage)

,indicator,records,first_observation,latest_observation,first_available_date,latest_available_date
0,CPI,66,2024-01-01,2026-08-01,2024-02-13,2026-09-11
1,FED_RATE,32,2024-01-01,2026-08-01,2024-02-01,2026-09-01
2,REAL_GDP,34,2024-01-01,2026-04-01,2024-04-25,2026-08-26
3,UNEMPLOYMENT,38,2024-01-01,2026-08-01,2024-02-02,2026-09-04


In [3]:
prices = pd.read_sql_query("""
    SELECT date, ticker, close_price
    FROM daily_prices
    ORDER BY date, ticker
""", conn)

prices["date"] = pd.to_datetime(prices["date"])

price_matrix = (
    prices.pivot(
        index="date",
        columns="ticker",
        values="close_price"
    )
    .sort_index()
)

vintages = pd.read_sql_query("""
    SELECT *
    FROM economic_vintages
    ORDER BY indicator, available_date
""", conn)

vintages["observation_date"] = pd.to_datetime(
    vintages["observation_date"]
)

vintages["available_date"] = pd.to_datetime(
    vintages["available_date"]
)

print("Stock price matrix:", price_matrix.shape)
print("Economic vintage records:", len(vintages))

display(vintages.head())

Stock price matrix: (250, 5)
Economic vintage records: 170


,indicator,observation_date,available_date,vintage_end_date,value,source
0,CPI,2024-01-01,2024-02-13,2025-02-11,309.685,ALFRED
1,CPI,2024-02-01,2024-03-12,2025-02-11,311.054,ALFRED
2,CPI,2024-03-01,2024-04-10,2025-02-11,312.230,ALFRED
3,CPI,2024-04-01,2024-05-15,2025-02-11,313.207,ALFRED
4,CPI,2024-05-01,2024-06-12,2025-02-11,313.225,ALFRED


In [4]:
trading_dates = price_matrix.index

def economic_data_as_of(trading_date, indicator):
    eligible = vintages[
        (vintages["indicator"] == indicator)
        & (vintages["available_date"] < trading_date)
        & (vintages["observation_date"] < trading_date)
    ].copy()

    if eligible.empty:
        return np.nan

    latest_observation = eligible["observation_date"].max()

    latest_vintage = (
        eligible[
            eligible["observation_date"] == latest_observation
        ]
        .sort_values("available_date")
        .iloc[-1]
    )

    return latest_vintage["value"]


indicators = [
    "CPI",
    "FED_RATE",
    "REAL_GDP",
    "UNEMPLOYMENT"
]

economic_daily = pd.DataFrame(
    index=trading_dates
)

for indicator in indicators:
    economic_daily[indicator] = [
        economic_data_as_of(date, indicator)
        for date in trading_dates
    ]

economic_daily.index.name = "date"

display(economic_daily.tail(10))

,CPI,FED_RATE,REAL_GDP,UNEMPLOYMENT
date,,,,
2026-09-08,332.813,3.63,24269.613,4.1
2026-09-09,332.813,3.63,24269.613,4.1
2026-09-10,332.813,3.63,24269.613,4.1
2026-09-11,332.813,3.63,24269.613,4.1
2026-09-14,334.131,3.63,24269.613,4.1
2026-09-15,334.131,3.63,24269.613,4.1
2026-09-16,334.131,3.63,24269.613,4.1
2026-09-17,334.131,3.63,24269.613,4.1
2026-09-18,334.131,3.63,24269.613,4.1


In [5]:
economic_daily["inflation_yoy"] = (
    economic_daily["CPI"]
    / economic_daily["CPI"].shift(252)
    - 1
) * 100

economic_daily["gdp_change"] = (
    economic_daily["REAL_GDP"].pct_change(
        periods=63,
        fill_method=None
    )
) * 100

display(economic_daily.tail(10))

,CPI,FED_RATE,REAL_GDP,UNEMPLOYMENT,inflation_yoy,gdp_change
date,,,,,,
2026-09-08,332.813,3.63,24269.613,4.1,NaN,0.484241
2026-09-09,332.813,3.63,24269.613,4.1,NaN,0.484241
2026-09-10,332.813,3.63,24269.613,4.1,NaN,0.484241
2026-09-11,332.813,3.63,24269.613,4.1,NaN,0.484241
2026-09-14,334.131,3.63,24269.613,4.1,NaN,0.484241
2026-09-15,334.131,3.63,24269.613,4.1,NaN,0.484241
2026-09-16,334.131,3.63,24269.613,4.1,NaN,0.484241
2026-09-17,334.131,3.63,24269.613,4.1,NaN,0.484241
2026-09-18,334.131,3.63,24269.613,4.1,NaN,0.484241


In [6]:
test_date = pd.Timestamp("2026-03-02")

eligible_records = vintages[
    vintages["available_date"] < test_date
]

assert (
    eligible_records["available_date"] < test_date
).all()

print("PASS: No future economic releases included.")
print("Eligible records:", len(eligible_records))

display(
    eligible_records[
        eligible_records["indicator"] == "CPI"
    ].tail()
)

PASS: No future economic releases included.
Eligible records: 142


,indicator,observation_date,available_date,vintage_end_date,value,source
54,CPI,2025-08-01,2026-02-13,2026-09-24,323.291,ALFRED
55,CPI,2025-09-01,2026-02-13,2026-09-24,324.245,ALFRED
56,CPI,2025-11-01,2026-02-13,2026-09-24,325.063,ALFRED
57,CPI,2025-12-01,2026-02-13,2026-09-24,326.031,ALFRED
58,CPI,2026-01-01,2026-02-13,2026-09-24,326.588,ALFRED


In [7]:
def latest_growth_as_of(trading_date, indicator, months_back):
    eligible = vintages[
        (vintages["indicator"] == indicator)
        & (vintages["available_date"] < trading_date)
        & (vintages["observation_date"] < trading_date)
    ].copy()

    if eligible.empty:
        return np.nan

    # Keep the latest vintage available for each observation.
    eligible = (
        eligible.sort_values("available_date")
        .drop_duplicates("observation_date", keep="last")
        .set_index("observation_date")["value"]
        .sort_index()
    )

    latest_date = eligible.index.max()
    previous_date = latest_date - pd.DateOffset(months=months_back)

    if previous_date not in eligible.index:
        return np.nan

    current = eligible.loc[latest_date]
    previous = eligible.loc[previous_date]

    if previous <= 0:
        return np.nan

    return (current / previous - 1) * 100


economic_daily["inflation_yoy"] = [
    latest_growth_as_of(date, "CPI", 12)
    for date in economic_daily.index
]

economic_daily["gdp_growth_yoy"] = [
    latest_growth_as_of(date, "REAL_GDP", 12)
    for date in economic_daily.index
]

display(
    economic_daily[
        ["inflation_yoy", "gdp_growth_yoy",
         "UNEMPLOYMENT", "FED_RATE"]
    ].tail(10)
)

,inflation_yoy,gdp_growth_yoy,UNEMPLOYMENT,FED_RATE
date,,,,
2026-09-08,3.303856,2.097672,4.1,3.63
2026-09-09,3.303856,2.097672,4.1,3.63
2026-09-10,3.303856,2.097672,4.1,3.63
2026-09-11,3.303856,2.097672,4.1,3.63
2026-09-14,3.353016,2.097672,4.1,3.63
2026-09-15,3.353016,2.097672,4.1,3.63
2026-09-16,3.353016,2.097672,4.1,3.63
2026-09-17,3.353016,2.097672,4.1,3.63
2026-09-18,3.353016,2.097672,4.1,3.63


In [8]:
daily_returns = price_matrix.pct_change().dropna()

portfolio_returns = daily_returns.mean(axis=1)

analysis = economic_daily.join(
    portfolio_returns.rename("portfolio_return"),
    how="inner"
)

analysis["inflation_regime"] = np.where(
    analysis["inflation_yoy"].isna(),
    "Insufficient history",
    np.where(
        analysis["inflation_yoy"] >= 3,
        "Inflation >= 3%",
        "Inflation < 3%"
    )
)

analysis["growth_regime"] = np.where(
    analysis["gdp_growth_yoy"].isna(),
    "Insufficient history",
    np.where(
        analysis["gdp_growth_yoy"] >= 2,
        "GDP growth >= 2%",
        "GDP growth < 2%"
    )
)

analysis["economic_regime"] = (
    analysis["inflation_regime"]
    + " / "
    + analysis["growth_regime"]
)

regime_results = (
    analysis.groupby("economic_regime")
    ["portfolio_return"]
    .agg(
        observations="count",
        average_daily_return="mean",
        daily_volatility="std"
    )
)

regime_results[
    ["average_daily_return", "daily_volatility"]
] *= 100

display(regime_results.round(3))

,observations,average_daily_return,daily_volatility
economic_regime,,,
Inflation < 3% / GDP growth < 2%,1,-0.104,NaN
Inflation < 3% / GDP growth >= 2%,98,-0.050,1.243
Inflation >= 3% / GDP growth < 2%,14,0.342,0.925
Inflation >= 3% / GDP growth >= 2%,136,0.118,1.100


In [9]:
assert analysis.index.is_monotonic_increasing
assert analysis.index.is_unique
assert analysis["portfolio_return"].notna().all()

print("PASS: Trading dates are ordered and unique.")
print("PASS: Portfolio returns contain no missing values.")
print("Trading sessions:", len(analysis))

print("\nEconomic-regime coverage:")
display(
    analysis["economic_regime"]
    .value_counts()
    .rename_axis("regime")
    .reset_index(name="trading_sessions")
)

PASS: Trading dates are ordered and unique.
PASS: Portfolio returns contain no missing values.
Trading sessions: 249

Economic-regime coverage:


,regime,trading_sessions
0,Inflation >= 3% / GDP growth >= 2%,136
1,Inflation < 3% / GDP growth >= 2%,98
2,Inflation >= 3% / GDP growth < 2%,14
3,Inflation < 3% / GDP growth < 2%,1


In [10]:
# Match information known before today's session
# with the return from today's close to the next close.

analysis["next_session_return"] = (
    portfolio_returns.shift(-1)
    .reindex(analysis.index)
)

valid_regimes = (
    analysis["economic_regime"]
    .value_counts()
    .loc[lambda counts: counts >= 20]
    .index
)

comparison = (
    analysis[
        analysis["economic_regime"].isin(valid_regimes)
    ]
    .dropna(subset=["next_session_return"])
    .groupby("economic_regime")["next_session_return"]
    .agg(
        sessions="count",
        average_return="mean",
        volatility="std"
    )
)

comparison[["average_return", "volatility"]] *= 100

display(comparison.round(3))

,sessions,average_return,volatility
economic_regime,,,
Inflation < 3% / GDP growth >= 2%,98,-0.043,1.245
Inflation >= 3% / GDP growth >= 2%,135,0.110,1.100


## Day 28 — Findings and limitations

- Reconstructed economic information using historical
  observation and availability dates.
- Applied a conservative next-session timing convention.
- Validated 249 historical portfolio-return observations.
- Compared economic regimes only where sample coverage
  was sufficient for exploratory analysis.

### Limitations

The historical sample is short and economic-regime
observations are unevenly distributed. Economic data
vintages may also have incomplete revision histories.

The results describe historical associations, not
predictive trading signals. No live trading is performed.